# Multimodal RAG with EURI and Free Local Embeddings

This notebook mirrors the multimodal RAG workflow while using EURI's OpenAI-compatible API for summaries and answers.

PDF tables, figures, captions, and images are preserved. Embeddings run locally with `BAAI/bge-small-en-v1.5`.

## Setup

Install macOS system dependencies once in Terminal:

```bash
brew install poppler tesseract libmagic
uv add unstructured langchain-chroma langchain-community langchain-huggingface sentence-transformers python-dotenv
```

Your `.env` should contain `EURI_API_KEY`. Optional settings are `EURI_BASE_URL` and `EURI_MODEL`.

In [ ]:
import importlib.util
import json
import os
import sys
from pathlib import Path
from urllib.error import HTTPError
from urllib.request import Request, urlopen

from dotenv import load_dotenv
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings
from unstructured.chunking.title import chunk_by_title
from unstructured.partition.pdf import partition_pdf

load_dotenv()

current_directory = Path.cwd()
PROJECT_DIRECTORY = (
    current_directory / 'rag-for-beginners'
    if (current_directory / 'rag-for-beginners' / 'docs').is_dir()
    else current_directory
)
PDF_PATH = PROJECT_DIRECTORY / 'docs' / '66---16-06-2021-02-48-53.pdf'
PERSIST_DIRECTORY = PROJECT_DIRECTORY / 'db' / 'chroma_multimodal_euri_bge'
EXPORT_DIRECTORY = PROJECT_DIRECTORY / 'multimodal_exports'
EMBEDDING_MODEL = 'BAAI/bge-small-en-v1.5'
EURI_BASE_URL = os.getenv('EURI_BASE_URL', 'https://api.euron.one/api/v1/euri')
EURI_MODEL = os.getenv('EURI_MODEL', 'gpt-4.1-mini')
print(f'EURI endpoint: {EURI_BASE_URL}')
print(f'EURI model: {EURI_MODEL}')

## 1. Partition the PDF into text, tables, figures, and captions

In [ ]:
if not PDF_PATH.is_file():
    raise FileNotFoundError(f'PDF file not found: {PDF_PATH}')

elements = partition_pdf(
    filename=str(PDF_PATH),
    strategy='hi_res',
    infer_table_structure=True,
    extract_image_block_types=['Image'],
    extract_image_block_to_payload=True,
)
images = [element for element in elements if getattr(element, 'category', '') == 'Image']
tables = [element for element in elements if getattr(element, 'category', '') == 'Table']
print(f'Extracted {len(elements)} elements | Images: {len(images)} | Tables: {len(tables)}')

## 2. Create structure-aware chunks and preserve multimodal content

In [ ]:
chunks = chunk_by_title(
    elements,
    max_characters=3000,
    new_after_n_chars=2400,
    combine_text_under_n_chars=500,
)

def separate_content_types(chunk):
    content = {'text': chunk.text, 'tables': [], 'images': [], 'types': ['text']}
    original_elements = getattr(getattr(chunk, 'metadata', None), 'orig_elements', [])
    for element in original_elements:
        element_type = type(element).__name__
        if element_type == 'Table':
            content['types'].append('table')
            metadata = getattr(element, 'metadata', None)
            content['tables'].append(getattr(metadata, 'text_as_html', getattr(element, 'text', '')))
        elif element_type == 'Image':
            image_base64 = getattr(getattr(element, 'metadata', None), 'image_base64', None)
            if image_base64:
                content['types'].append('image')
                content['images'].append(image_base64)
    content['types'] = sorted(set(content['types']))
    return content

print(f'Created {len(chunks)} structure-aware chunks')
print(separate_content_types(chunks[0])['types'])

## 3. Call EURI through its OpenAI-compatible chat endpoint

In [ ]:
def call_euri(prompt_text, images=None):
    euri_api_key = os.getenv('EURI_API_KEY')
    if not euri_api_key:
        raise ValueError('EURI_API_KEY is not set in the environment or .env file')

    content = [{'type': 'text', 'text': prompt_text}]
    for image_base64 in images or []:
        content.append({
            'type': 'image_url',
            'image_url': {'url': f'data:image/jpeg;base64,{image_base64}'},
        })

    endpoint = EURI_BASE_URL.rstrip('/') + '/chat/completions'
    request = Request(
        endpoint,
        data=json.dumps({
            'model': EURI_MODEL,
            'messages': [{'role': 'user', 'content': content}],
            'temperature': 0,
            'max_tokens': 1200,
        }).encode('utf-8'),
        headers={
            'Authorization': f'Bearer {euri_api_key}',
            'Content-Type': 'application/json',
            'User-Agent': 'genai-euri-multimodal-rag/1.0',
        },
        method='POST',
    )
    try:
        with urlopen(request) as response:
            result = json.load(response)
    except HTTPError as error:
        details = error.read().decode('utf-8', errors='replace')
        raise RuntimeError(f'EURI request failed with HTTP {error.code}: {details}') from error
    return result['choices'][0]['message']['content']

In [ ]:
def create_searchable_summary(content):
    prompt = f'''Create a detailed searchable description of this PDF chunk.
Cover key facts, numbers, concepts, table values, figure captions, and likely questions.

TEXT:
{content['text']}

TABLES:
{json.dumps(content['tables'], ensure_ascii=False)}

Detected image count: {len(content['images'])}

Return only the searchable description.'''
    if content['tables'] or content['images']:
        return call_euri(prompt, content['images'])
    return content['text']

processed_chunks = []
for index, chunk in enumerate(chunks, 1):
    content = separate_content_types(chunk)
    enhanced_content = create_searchable_summary(content)
    processed_chunks.append(Document(
        page_content=enhanced_content,
        metadata={
            'source': str(PDF_PATH),
            'chunk_id': index,
            'original_content': json.dumps({
                'raw_text': content['text'],
                'tables_html': content['tables'],
                'images_base64': content['images'],
            }, ensure_ascii=False),
        },
    ))
    if index % 10 == 0 or index == len(chunks):
        print(f'Processed {index}/{len(chunks)} chunks')

## 4. Embed locally and store in Chroma

In [ ]:
embeddings = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL,
    model_kwargs={'device': 'cpu'},
    encode_kwargs={'normalize_embeddings': True},
)

db = Chroma.from_documents(
    documents=processed_chunks,
    embedding=embeddings,
    persist_directory=str(PERSIST_DIRECTORY),
    collection_metadata={'hnsw:space': 'cosine'},
)
print(f'Chroma database ready at {PERSIST_DIRECTORY}')

## 5. Retrieve multimodal context and ask EURI

In [ ]:
def export_documents(documents, filename):
    EXPORT_DIRECTORY.mkdir(parents=True, exist_ok=True)
    data = []
    for index, document in enumerate(documents, 1):
        data.append({
            'chunk_id': index,
            'enhanced_content': document.page_content,
            'metadata': {
                'source': document.metadata.get('source'),
                'original_content': json.loads(document.metadata.get('original_content', '{}')),
            },
        })
    output_path = EXPORT_DIRECTORY / filename
    output_path.write_text(json.dumps(data, indent=2, ensure_ascii=False), encoding='utf-8')
    print(f'Exported {len(data)} chunks to {output_path}')

question = 'What are the main ideas, important facts, and key tables or figures in this document?'
retrieved_chunks = db.similarity_search(question, k=5)
export_documents(retrieved_chunks, 'euri_multimodal_rag_results.json')

prompt_parts = [f'''Answer this question using only the retrieved PDF content: {question}

Use the text, tables, captions, and images. If evidence is insufficient, say you do not know.

RETRIEVED CONTENT:
''']
retrieved_images = []
for index, document in enumerate(retrieved_chunks, 1):
    original = json.loads(document.metadata.get('original_content', '{}'))
    prompt_parts.append(f'--- Chunk {index} ---\nTEXT:\n{original.get("raw_text", "")}\n')
    for table_index, table in enumerate(original.get('tables_html', []), 1):
        prompt_parts.append(f'TABLE {table_index}:\n{table}\n')
    retrieved_images.extend(original.get('images_base64', []))
prompt_parts.append(f'\nRetrieved image payloads: {len(retrieved_images)}\nGive a clear answer with important facts and numbers.')

print(call_euri('\n'.join(prompt_parts), retrieved_images))